# 01 — Data ingestion and leakage audit

## What is this notebook for?

Before training a model, we first define a **fair prediction question**.

Northbridge wants an early warning:

> **“This ticket looks likely to breach its SLA. Can we intervene now?”**

The prediction point in this project is **T1: immediately after the ticket has been created and routed to an agent/team**.

This notebook asks five simple questions:

1. What data do we have?
2. How do the three tables connect?
3. How common is breach?
4. Which columns are legal at T1?
5. Which columns would let the model see the future?

Think of this as checking the measuring instruments before starting an experiment.

In [ ]:
import sys
from pathlib import Path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
print("Project root:", project_root)
import warnings
warnings.filterwarnings("ignore")

from src.config import data_dir
from src.data.load import load_source_tables 
from src.data.load import build_ticket_master
from src.data.audit import audit_frame

Project root: c:\Users\woro_\OneDrive\DS_Projects\northbridge-sla-breach


In [3]:
tickets, agents, clients = load_source_tables(data_dir())
print(f"Tickets: {tickets.shape[0]:,} rows × {tickets.shape[1]} columns")
print(f"Agents:  {agents.shape[0]:,} rows × {agents.shape[1]} columns")
print(f"Clients: {clients.shape[0]:,} rows × {clients.shape[1]} columns")

Tickets: 3,500 rows × 16 columns
Agents:  999 rows × 8 columns
Clients: 999 rows × 10 columns


c:\Users\woro_\OneDrive\DS_Projects\northbridge-sla-breach\src\data\load.py:27: UserWarning: Parsing dates in %Y-%m-%d %H:%M format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  tickets[col] = pd.to_datetime(tickets[col], dayfirst=True, errors="coerce")
c:\Users\woro_\OneDrive\DS_Projects\northbridge-sla-breach\src\data\load.py:27: UserWarning: Parsing dates in %Y-%m-%d %H:%M format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  tickets[col] = pd.to_datetime(tickets[col], dayfirst=True, errors="coerce")
c:\Users\woro_\OneDrive\DS_Projects\northbridge-sla-breach\src\data\load.py:27: UserWarning: Parsing dates in %Y-%m-%d %H:%M format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  tickets[col] = pd.to_datetime(tickets[col], dayfirst=True, errors="coerce")
c:\Users\woro_\OneDrive\DS_Projects\northbridge-sla-breach\src\data

### Look at a tiny slice, not all 3,500 tickets

A few rows are enough to understand what one observation looks like.

In [4]:
tickets[["TicketID","CategoryID","PriorityID","Channel","CreatedAt","SLADueAt","SLABreached"]].head()

,TicketID,CategoryID,PriorityID,Channel,CreatedAt,SLADueAt,SLABreached
0,1,5,3,Phone,2024-07-24 23:41:00,2024-07-27 23:41:00,0
1,2,2,1,Portal,2024-11-06 23:45:00,2024-11-07 07:45:00,0
2,3,3,3,Portal,2024-08-01 05:39:00,2024-08-04 05:39:00,0
3,4,1,2,Form,2024-12-03 13:22:00,2024-12-04 01:22:00,1
4,5,3,2,Phone,2023-02-11 03:56:00,2023-02-12 03:56:00,0


## 2. What do the important columns mean?

| Column | Plain-language meaning |
|---|---|
| `PriorityID` | 1 Critical, 2 High, 3 Standard, 4 Low |
| `CategoryID` | request family |
| `Channel` | Phone, Email, Form or Portal |
| `CreatedAt` | when the request entered the system |
| `SLADueAt` | contractual deadline, if already known |
| `FirstResponseAt` | when someone first responded |
| `ResolvedAt` | when the case was resolved |
| `Status` | recorded workflow state |
| `ResolutionNotes` | what was eventually done |
| `SLABreached` | the answer we want to predict |

The key distinction is:

> **Known now is different from known eventually.**

## 3. Join the business context

A ticket alone does not tell us the contract or resource handling it.

We add:

- `ContractTier`, `SLACreditClause` from Clients;
- `TeamID`, `Hub`, `Role`, `DailyCapacity` from Agents.

In [11]:
# Build the master table by joining the Facts table (Tickets) with the filtered dimension tables (Agents and Clients)

master = build_ticket_master({"Tickets": tickets, "Agents": agents, "Clients": clients})
print(f"Joined table: {master.shape[0]:,} rows × {master.shape[1]} columns")
master[["TicketID","PriorityID","CategoryID","Channel","ContractTier","TeamID","Hub","DailyCapacity","SLABreached"]].head()

Joined table: 3,500 rows × 21 columns


,TicketID,PriorityID,CategoryID,Channel,ContractTier,TeamID,Hub,DailyCapacity,SLABreached
0,1,3,5,Phone,Standard,8.000,Edinburgh,16.000,0
1,2,1,2,Portal,Standard,1.000,Manchester,8.000,0
2,3,3,3,Portal,Standard,4.000,Leeds,21.000,0
3,4,2,1,Form,Premium,1.000,Manchester,27.000,1
4,5,2,3,Phone,Standard,4.000,Leeds,26.000,0


A model row now represents:
Request + Contract + Assigned resource
```
That is the business context available at T1.

## 4. The Time-Machine test

For every feature ask:

> **Could an operations analyst know this value at T1 without waiting for the future?**

The answer must be based on the real workflow, not on how predictive the column looks.

In [6]:
feature_boundary = pd.DataFrame([
    ["PriorityID","Yes","Known on the ticket"],
    ["CategoryID","Yes","Known on the ticket"],
    ["Channel","Yes","Known at intake"],
    ["ContractTier","Yes","Known from contract"],
    ["DailyCapacity","Yes*","Allowed because T1 is after assignment"],
    ["SLADueAt","Yes*","Only if deadline already exists at T1"],
    ["FirstResponseAt","No","Occurs after prediction"],
    ["ResolvedAt","No","Occurs after prediction"],
    ["Status","No","Final state may contain future information"],
    ["ResolutionNotes","No","Written after work is performed"],
    ["SLABreached","TARGET","This is the answer"],
], columns=["Field","Allowed at T1?","Reason"])
feature_boundary

,Field,Allowed at T1?,Reason
0,PriorityID,Yes,Known on the ticket
1,CategoryID,Yes,Known on the ticket
2,Channel,Yes,Known at intake
3,ContractTier,Yes,Known from contract
4,DailyCapacity,Yes*,Allowed because T1 is after assignment
5,SLADueAt,Yes*,Only if deadline already exists at T1
6,FirstResponseAt,No,Occurs after prediction
7,ResolvedAt,No,Occurs after prediction
8,Status,No,Final state may contain future information
9,ResolutionNotes,No,Written after work is performed


## 5. Run a Compact Data Audit

We check:

- duplicates;
- missingness;
- date coverage;
- target prevalence;
- future/outcome columns physically present in the source.

A future column may remain in the raw data for historical analysis. It simply must not be fed directly to the T1 model.

In [7]:
audit = audit_frame(master)
pd.Series({
    "Rows": audit["rows"],
    "Columns": audit["columns"],
    "Duplicate TicketIDs": audit["duplicate_ticket_ids"],
    "Breach prevalence": audit["target_prevalence"],
    "First ticket": audit["date_min"],
    "Last ticket": audit["date_max"],
})

Rows                                  3500
Columns                                 21
Duplicate TicketIDs                      0
Breach prevalence                    0.215
First ticket           2023-01-01 10:54:00
Last ticket            2025-03-31 07:56:00
dtype: object

In [8]:
print("Future/outcome columns present:")
print(audit["leakage_columns_present"])
print("\nFive columns with the most missingness:")
pd.Series(audit["missing_fraction"]).sort_values(ascending=False).head()

Future/outcome columns present:
['FirstResponseAt', 'ResolvedAt', 'Status', 'ResolutionNotes']

Five columns with the most missingness:


ResolvedAt        0.057
ResolutionNotes   0.057
FirstResponseAt   0.013
TicketReference   0.000
TicketID          0.000
dtype: float64

## 6. Class imbalance: why accuracy can mislead

If most tickets are on time, a useless model can achieve high accuracy by always saying **“no breach.”**

That is why later notebooks emphasise ROC-AUC, PR-AUC, recall and Lift@10%.

In [9]:
breach_rate = master["SLABreached"].mean()
print(f"Overall breach rate: {breach_rate:.1%}")
print(master["SLABreached"].value_counts().rename({0:"On time",1:"Breached"}))

Overall breach rate: 21.5%
SLABreached
On time     2746
Breached     754
Name: count, dtype: int64


In [10]:
lazy_accuracy = (master["SLABreached"] == 0).mean()
print(f'Accuracy of an "always on-time" rule: {lazy_accuracy:.1%}')
print("Recall for real breaches: 0.0%")

Accuracy of an "always on-time" rule: 78.5%
Recall for real breaches: 0.0%


## 7. Notebook 01 checkpoint

We have deliberately trained **nothing** yet.

Success here means the modelling boundary is trustworthy:

```text
Legal at T1                 Future / outcome
-----------                 ----------------
Priority                    First response
Category                    Resolution time
Channel                     Final status
Contract                    Resolution notes
Assigned resource           SLA breach outcome
Creation/SLA timing
```

**Next:** Notebook 02 asks whether the legal T1 features actually separate breached and non-breached tickets.